
# 01 - Create Auto Sales Delta Table

Goal:
Create a mock dealership sales dataset and store it as a Delta Table.

Business case:
Acme Motors wants a safe assistant that answers sales questions from Delta Tables instead of letting an LLM invent numbers.

Output table:
auto_sales_transactions

In [0]:
from datetime import date, datetime
from pyspark.sql.functions import col, current_timestamp, to_date
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    IntegerType,
    DoubleType,
    DateType
)

PROJECT_NAME = "acme_motors_assistant"
SOURCE_TABLE = "auto_sales_transactions"
AUDIT_TABLE = "assistant_query_audit"

TODAY = date(2026, 5, 23)

print(f"Project: {PROJECT_NAME}")
print(f"Source table: {SOURCE_TABLE}")
print(f"Audit table: {AUDIT_TABLE}")
print(f"Today: {TODAY}")

Project: acme_motors_assistant
Source table: auto_sales_transactions
Audit table: assistant_query_audit
Today: 2026-05-23


In [0]:
sales_data = [
    {
        "sale_id": "S001",
        "sale_date": date(2026, 5, 1),
        "dealership_id": "D001",
        "dealership_name": "Acme Motors North",
        "brand": "Ford",
        "product_model": "F150",
        "vehicle_type": "Truck",
        "sales_rep": "Ana Lopez",
        "customer_state": "TX",
        "quantity": 1,
        "sale_price": 52000.00,
        "discount_amount": 1500.00,
        "financing_type": "Finance"
    },
    {
        "sale_id": "S002",
        "sale_date": date(2026, 5, 3),
        "dealership_id": "D001",
        "dealership_name": "Acme Motors North",
        "brand": "Ford",
        "product_model": "F150",
        "vehicle_type": "Truck",
        "sales_rep": "Ana Lopez",
        "customer_state": "TX",
        "quantity": 1,
        "sale_price": 53500.00,
        "discount_amount": 1000.00,
        "financing_type": "Lease"
    },
    {
        "sale_id": "S003",
        "sale_date": date(2026, 5, 5),
        "dealership_id": "D002",
        "dealership_name": "Acme Motors South",
        "brand": "Chevrolet",
        "product_model": "Silverado",
        "vehicle_type": "Truck",
        "sales_rep": "Luis Garcia",
        "customer_state": "CA",
        "quantity": 1,
        "sale_price": 49000.00,
        "discount_amount": 1200.00,
        "financing_type": "Finance"
    },
    {
        "sale_id": "S004",
        "sale_date": date(2026, 5, 7),
        "dealership_id": "D003",
        "dealership_name": "Acme Motors East",
        "brand": "Ford",
        "product_model": "Mustang",
        "vehicle_type": "Coupe",
        "sales_rep": "Mike Johnson",
        "customer_state": "AZ",
        "quantity": 1,
        "sale_price": 44000.00,
        "discount_amount": 800.00,
        "financing_type": "Cash"
    },
    {
        "sale_id": "S005",
        "sale_date": date(2026, 5, 8),
        "dealership_id": "D002",
        "dealership_name": "Acme Motors South",
        "brand": "Toyota",
        "product_model": "Camry",
        "vehicle_type": "Sedan",
        "sales_rep": "Luis Garcia",
        "customer_state": "CA",
        "quantity": 1,
        "sale_price": 31000.00,
        "discount_amount": 500.00,
        "financing_type": "Finance"
    },
    {
        "sale_id": "S006",
        "sale_date": date(2026, 5, 10),
        "dealership_id": "D001",
        "dealership_name": "Acme Motors North",
        "brand": "Ford",
        "product_model": "F150",
        "vehicle_type": "Truck",
        "sales_rep": "Ana Lopez",
        "customer_state": "TX",
        "quantity": 1,
        "sale_price": 54000.00,
        "discount_amount": 900.00,
        "financing_type": "Finance"
    },
    {
        "sale_id": "S007",
        "sale_date": date(2026, 5, 11),
        "dealership_id": "D003",
        "dealership_name": "Acme Motors East",
        "brand": "Honda",
        "product_model": "Civic",
        "vehicle_type": "Sedan",
        "sales_rep": "Sarah Kim",
        "customer_state": "NV",
        "quantity": 1,
        "sale_price": 28000.00,
        "discount_amount": 300.00,
        "financing_type": "Lease"
    },
    {
        "sale_id": "S008",
        "sale_date": date(2026, 5, 13),
        "dealership_id": "D002",
        "dealership_name": "Acme Motors South",
        "brand": "Chevrolet",
        "product_model": "Silverado",
        "vehicle_type": "Truck",
        "sales_rep": "Luis Garcia",
        "customer_state": "CA",
        "quantity": 1,
        "sale_price": 50500.00,
        "discount_amount": 1000.00,
        "financing_type": "Finance"
    },
    {
        "sale_id": "S009",
        "sale_date": date(2026, 5, 15),
        "dealership_id": "D001",
        "dealership_name": "Acme Motors North",
        "brand": "Ford",
        "product_model": "F150",
        "vehicle_type": "Truck",
        "sales_rep": "Ana Lopez",
        "customer_state": "TX",
        "quantity": 1,
        "sale_price": 52500.00,
        "discount_amount": 1100.00,
        "financing_type": "Cash"
    },
    {
        "sale_id": "S010",
        "sale_date": date(2026, 5, 17),
        "dealership_id": "D003",
        "dealership_name": "Acme Motors East",
        "brand": "Ford",
        "product_model": "Explorer",
        "vehicle_type": "SUV",
        "sales_rep": "Mike Johnson",
        "customer_state": "AZ",
        "quantity": 1,
        "sale_price": 47000.00,
        "discount_amount": 700.00,
        "financing_type": "Finance"
    },
    {
        "sale_id": "S011",
        "sale_date": date(2026, 5, 18),
        "dealership_id": "D004",
        "dealership_name": "Acme Motors West",
        "brand": "Toyota",
        "product_model": "RAV4",
        "vehicle_type": "SUV",
        "sales_rep": "Carlos Rivera",
        "customer_state": "WA",
        "quantity": 1,
        "sale_price": 39000.00,
        "discount_amount": 650.00,
        "financing_type": "Lease"
    },
    {
        "sale_id": "S012",
        "sale_date": date(2026, 5, 20),
        "dealership_id": "D001",
        "dealership_name": "Acme Motors North",
        "brand": "Ford",
        "product_model": "F150",
        "vehicle_type": "Truck",
        "sales_rep": "Ana Lopez",
        "customer_state": "TX",
        "quantity": 1,
        "sale_price": 54500.00,
        "discount_amount": 950.00,
        "financing_type": "Finance"
    },
    {
        "sale_id": "S013",
        "sale_date": date(2026, 4, 12),
        "dealership_id": "D001",
        "dealership_name": "Acme Motors North",
        "brand": "Ford",
        "product_model": "F150",
        "vehicle_type": "Truck",
        "sales_rep": "Ana Lopez",
        "customer_state": "TX",
        "quantity": 1,
        "sale_price": 51500.00,
        "discount_amount": 1300.00,
        "financing_type": "Finance"
    },
    {
        "sale_id": "S014",
        "sale_date": date(2026, 4, 20),
        "dealership_id": "D002",
        "dealership_name": "Acme Motors South",
        "brand": "Chevrolet",
        "product_model": "Silverado",
        "vehicle_type": "Truck",
        "sales_rep": "Luis Garcia",
        "customer_state": "CA",
        "quantity": 1,
        "sale_price": 48500.00,
        "discount_amount": 1100.00,
        "financing_type": "Cash"
    },
    {
        "sale_id": "S015",
        "sale_date": date(2026, 1, 10),
        "dealership_id": "D004",
        "dealership_name": "Acme Motors West",
        "brand": "Honda",
        "product_model": "Accord",
        "vehicle_type": "Sedan",
        "sales_rep": "Carlos Rivera",
        "customer_state": "WA",
        "quantity": 1,
        "sale_price": 34000.00,
        "discount_amount": 400.00,
        "financing_type": "Finance"
    }
]

print(f"Mock sales records: {len(sales_data)}")

Mock sales records: 15


In [0]:
# Define schema

sales_schema = StructType([
    StructField("sale_id", StringType(), False),
    StructField("sale_date", DateType(), False),
    StructField("dealership_id", StringType(), False),
    StructField("dealership_name", StringType(), False),
    StructField("brand", StringType(), False),
    StructField("product_model", StringType(), False),
    StructField("vehicle_type", StringType(), False),
    StructField("sales_rep", StringType(), False),
    StructField("customer_state", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("sale_price", DoubleType(), False),
    StructField("discount_amount", DoubleType(), False),
    StructField("financing_type", StringType(), False)
]) 

In [0]:
sales_df = spark.createDataFrame(sales_data, schema=sales_schema)
display(sales_df)

sale_id,sale_date,dealership_id,dealership_name,brand,product_model,vehicle_type,sales_rep,customer_state,quantity,sale_price,discount_amount,financing_type
S001,2026-05-01,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52000.0,1500.0,Finance
S002,2026-05-03,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,53500.0,1000.0,Lease
S003,2026-05-05,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,49000.0,1200.0,Finance
S004,2026-05-07,D003,Acme Motors East,Ford,Mustang,Coupe,Mike Johnson,AZ,1,44000.0,800.0,Cash
S005,2026-05-08,D002,Acme Motors South,Toyota,Camry,Sedan,Luis Garcia,CA,1,31000.0,500.0,Finance
S006,2026-05-10,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,54000.0,900.0,Finance
S007,2026-05-11,D003,Acme Motors East,Honda,Civic,Sedan,Sarah Kim,NV,1,28000.0,300.0,Lease
S008,2026-05-13,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,50500.0,1000.0,Finance
S009,2026-05-15,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52500.0,1100.0,Cash
S010,2026-05-17,D003,Acme Motors East,Ford,Explorer,SUV,Mike Johnson,AZ,1,47000.0,700.0,Finance


In [0]:
# Enrichment

sales_enriched_df = (
    sales_df
    .withColumn("created_at", current_timestamp())
)

display(sales_enriched_df)

sale_id,sale_date,dealership_id,dealership_name,brand,product_model,vehicle_type,sales_rep,customer_state,quantity,sale_price,discount_amount,financing_type,created_at
S001,2026-05-01,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52000.0,1500.0,Finance,2026-05-23T21:09:00.378Z
S002,2026-05-03,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,53500.0,1000.0,Lease,2026-05-23T21:09:00.378Z
S003,2026-05-05,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,49000.0,1200.0,Finance,2026-05-23T21:09:00.378Z
S004,2026-05-07,D003,Acme Motors East,Ford,Mustang,Coupe,Mike Johnson,AZ,1,44000.0,800.0,Cash,2026-05-23T21:09:00.378Z
S005,2026-05-08,D002,Acme Motors South,Toyota,Camry,Sedan,Luis Garcia,CA,1,31000.0,500.0,Finance,2026-05-23T21:09:00.378Z
S006,2026-05-10,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,54000.0,900.0,Finance,2026-05-23T21:09:00.378Z
S007,2026-05-11,D003,Acme Motors East,Honda,Civic,Sedan,Sarah Kim,NV,1,28000.0,300.0,Lease,2026-05-23T21:09:00.378Z
S008,2026-05-13,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,50500.0,1000.0,Finance,2026-05-23T21:09:00.378Z
S009,2026-05-15,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52500.0,1100.0,Cash,2026-05-23T21:09:00.378Z
S010,2026-05-17,D003,Acme Motors East,Ford,Explorer,SUV,Mike Johnson,AZ,1,47000.0,700.0,Finance,2026-05-23T21:09:00.378Z


In [0]:
# Data quality checks

total_records = sales_enriched_df.count()
distinct_sale_ids = sales_enriched_df.select("sale_id").distinct().count()

print(f"Total records: {total_records}")
print(f"Distinct sale IDs: {distinct_sale_ids}")

# Check for empty dataset
if total_records == 0:
    raise ValueError("Sales dataset is empty.")

# Check for duplicated records
if total_records != distinct_sale_ids:
    raise ValueError("Duplicate sale_id values detected.")

invalid_amounts = sales_enriched_df.filter(
    (col("sale_price") <= 0) | 
    (col("quantity") <= 0)
)

invalid_count = invalid_amounts.count()

print(f"Invalid amount records: {invalid_count}")

if invalid_count > 0:
    display(invalid_amounts)
    raise ValueError("Invalid sale_price or quantity detected.")


Total records: 15
Distinct sale IDs: 15
Invalid amount records: 0


In [0]:
# Saves as delta table
sales_enriched_df.write.format("delta").mode("overwrite").saveAsTable(SOURCE_TABLE)

print(f"Delta table created: {SOURCE_TABLE}")

Delta table created: auto_sales_transactions


In [0]:
# Read table from databricks
saved_sales_df = spark.table(SOURCE_TABLE)

display(saved_sales_df)

sale_id,sale_date,dealership_id,dealership_name,brand,product_model,vehicle_type,sales_rep,customer_state,quantity,sale_price,discount_amount,financing_type,created_at
S001,2026-05-01,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52000.0,1500.0,Finance,2026-05-23T21:11:25.384Z
S002,2026-05-03,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,53500.0,1000.0,Lease,2026-05-23T21:11:25.384Z
S003,2026-05-05,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,49000.0,1200.0,Finance,2026-05-23T21:11:25.384Z
S004,2026-05-07,D003,Acme Motors East,Ford,Mustang,Coupe,Mike Johnson,AZ,1,44000.0,800.0,Cash,2026-05-23T21:11:25.384Z
S005,2026-05-08,D002,Acme Motors South,Toyota,Camry,Sedan,Luis Garcia,CA,1,31000.0,500.0,Finance,2026-05-23T21:11:25.384Z
S006,2026-05-10,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,54000.0,900.0,Finance,2026-05-23T21:11:25.384Z
S007,2026-05-11,D003,Acme Motors East,Honda,Civic,Sedan,Sarah Kim,NV,1,28000.0,300.0,Lease,2026-05-23T21:11:25.384Z
S008,2026-05-13,D002,Acme Motors South,Chevrolet,Silverado,Truck,Luis Garcia,CA,1,50500.0,1000.0,Finance,2026-05-23T21:11:25.384Z
S009,2026-05-15,D001,Acme Motors North,Ford,F150,Truck,Ana Lopez,TX,1,52500.0,1100.0,Cash,2026-05-23T21:11:25.384Z
S010,2026-05-17,D003,Acme Motors East,Ford,Explorer,SUV,Mike Johnson,AZ,1,47000.0,700.0,Finance,2026-05-23T21:11:25.384Z


In [0]:
%sql
-- Get sales by model
SELECT
  product_model,
  SUM(quantity) AS units_sold,
  SUM(sale_price * quantity) AS total_revenue
FROM auto_sales_transactions
GROUP BY product_model
ORDER BY units_sold DESC, total_revenue DESC;

product_model,units_sold,total_revenue
F150,6,318000.0
Silverado,3,148000.0
Explorer,1,47000.0
Mustang,1,44000.0
RAV4,1,39000.0
Accord,1,34000.0
Camry,1,31000.0
Civic,1,28000.0


In [0]:
%sql
-- Get F150 sales this month
SELECT
  product_model,
  SUM(quantity) AS units_sold,
  SUM(sale_price * quantity) AS total_revenue
FROM auto_sales_transactions
WHERE product_model = 'F150'
  AND sale_date >= DATE('2026-05-01')
  AND sale_date <= DATE('2026-05-23')
GROUP BY product_model;

product_model,units_sold,total_revenue
F150,5,266500.0


In [0]:
%sql
-- Get sales by sales rep
SELECT
  sales_rep,
  SUM(quantity) AS units_sold,
  SUM(sale_price * quantity) AS total_revenue
FROM auto_sales_transactions
WHERE sale_date >= DATE('2026-05-01')
  AND sale_date <= DATE('2026-05-23')
GROUP BY sales_rep
ORDER BY units_sold DESC, total_revenue DESC;

sales_rep,units_sold,total_revenue
Ana Lopez,5,266500.0
Luis Garcia,3,130500.0
Mike Johnson,2,91000.0
Carlos Rivera,1,39000.0
Sarah Kim,1,28000.0


In [0]:
%sql
-- Get sales by state
SELECT
  customer_state,
  SUM(quantity) AS units_sold,
  SUM(sale_price * quantity) AS total_revenue
FROM auto_sales_transactions
WHERE sale_date >= DATE('2026-05-01')
  AND sale_date <= DATE('2026-05-23')
GROUP BY customer_state
ORDER BY total_revenue DESC;

customer_state,units_sold,total_revenue
TX,5,266500.0
CA,3,130500.0
AZ,2,91000.0
WA,1,39000.0
NV,1,28000.0
